### ======================================================
Consulting Challenge - TalentHub

Exploracion inicial del dataset seleccionado

Autor: Agustin Amarilla


### ===============================================================================

# Consulting Challenge - TalentHub

## Objetivo del proyecto

Analizar un conjunto de ofertas laborales relacionadas con el perfil de Data Analyst para identificar las habilidades más demandadas por el mercado.

## Objetivo de negocio

TalentHub desea diseñar un nuevo bootcamp alineado con las necesidades actuales de las empresas. Para ello, se analizarán las vacantes disponibles con el fin de identificar las tecnologías, competencias y perfiles más solicitados, permitiendo elaborar recomendaciones basadas en datos.

======================================================
Etapa 1 - Carga de datos
======================================================

In [ ]:
import pandas as pd 
habilidades = pd.read_csv('C:/Users/Cipher/Documents/ComIt/consulting-challenge/data/raw/skills_rows.csv')
vacantes = pd.read_csv('C:/Users/Cipher/Documents/ComIt/consulting-challenge/data/raw/vacancies_rows.csv')
vacantes_habilidades = pd.read_csv('C:/Users/Cipher/Documents/ComIt/consulting-challenge/data/raw/vacancy_skills_rows.csv')
df = pd.read_csv('C:/Users/Cipher/Documents/ComIt/consulting-challenge/data/processed/vacantes_completas.csv')

======================================================
Etapa 2 - Exploracion inicial
======================================================

In [ ]:
print("VACANTES")
print(vacantes.shape)

print("HABILIDADES")
print(habilidades.shape)

print("VACANTES_HABILIDADES")
print(vacantes_habilidades.shape)

vacantes.head

In [ ]:
print("Columnas de vacantes:")
print(vacantes.columns.tolist())

print("\nColumnas de habilidades:")
print(habilidades.columns.tolist())

print("\nColumnas de vacantes_habilidades:")
print(vacantes_habilidades.columns.tolist())

In [ ]:
print("Tipos de datos de vacantes:")
print(vacantes.dtypes)

print("\nTipos de datos de habilidades:")
print(habilidades.dtypes)

print("\nTipos de datos de vacantes_habilidades:")
print(vacantes_habilidades.dtypes)

In [ ]:
print("INFORMACIÓN DE VACANTES")
vacantes.info()

print("\nINFORMACIÓN DE HABILIDADES")
habilidades.info()

print("\nINFORMACIÓN DE VACANTES-HABILIDADES")
vacantes_habilidades.info()

### Verificación de tipos de datos

Se revisaron los tipos de datos de cada tabla para asegurar que fueran adecuados para el análisis.

Se realizaron las siguientes conversiones:

- `created_at` → `datetime`
- `published_at` → `datetime`

El resto de las columnas ya presentaban un tipo de dato consistente con la información que almacenan (identificadores numéricos, variables categóricas y valores numéricos).

    La columna "description_raw" y "experience_years" presentan valores nulos; sin embargo, no afecta el análisis realizado, ya que no fueron utilizadas en el proceso de exploración ni en la generación de insights.

======================================================
Etapa 3 - Diagnostico de calidad de los datos
======================================================

Valores Nulos

In [ ]:
print("Valores nulos en VACANTES")
print(vacantes.isnull().sum())

print("\nValores nulos en HABILIDADES")
print(habilidades.isnull().sum())

print("\nValores nulos en VACANTES_HABILIDADES")
print(vacantes_habilidades.isnull().sum())

In [ ]:
print("Duplicados en Vacantes:", vacantes.duplicated().sum())
print("Duplicados en Habilidades:", habilidades.duplicated().sum())
print("Duplicados en Vacantes-Habilidades:", vacantes_habilidades.duplicated().sum())

In [ ]:
print("IDs únicos en Vacantes:", vacantes["id"].is_unique)
print("IDs únicos en Habilidades:", habilidades["id"].is_unique)

In [ ]:
vacantes.describe(include="all")

======================================================
Exploracion de variables
======================================================

In [ ]:
print("Puestos más frecuentes:\n")
print(vacantes["title"].value_counts().head(15))

In [ ]:
vacantes["experience_level"] = (
    vacantes["experience_level"]
    .fillna("Not specified")
)
vacantes["experience_level"].isna().sum()


In [ ]:
print("Niveles de experiencia:\n")
print(vacantes["experience_level"].value_counts(dropna=False))

In [ ]:
vacantes["location"] = vacantes["location"].fillna("No especificada")
print("Ubicaciones más frecuentes:\n")
print(vacantes["location"].value_counts().head(20))


In [ ]:
print(vacantes["experience_years"].describe())
print(vacantes["experience_years"].value_counts().sort_index())

======================================================
Etapa 4 - Limpieza de datos
======================================================

In [ ]:
# hacemos limpieza de los datos de texto eliminando espacios en blanco al inicio y al final de cada cadena de texto
columnas_texto = [
    "title",
    "company",
    "location",
    "experience_level",
    "description_raw"
]

for columna in columnas_texto:
    vacantes[columna] = vacantes[columna].str.strip()

In [ ]:
#  revisamos si existen diferencias solo por mayusculas
vacantes["location"].value_counts().head(30)

In [ ]:
vacantes["title_normalizado"] = (
    vacantes["title"]
    .str.strip()
    .str.lower()
)

print("Títulos originales:", vacantes["title"].nunique())
print("Títulos normalizados:", vacantes["title_normalizado"].nunique())

In [ ]:
top_puestos = (
    vacantes["title_normalizado"]
    .value_counts()
    .head(15)
)

top_puestos

======================================================
Se puede observar que hay muchos puestos iguales, asi que vamos a tomar una decicon logica y agruparlos en una nueva columna
======================================================

In [ ]:
def normalizar_puesto(titulo):
    titulo = titulo.lower().strip()

    if "senior data analyst" in titulo or "sr. data analyst" in titulo:
        return "Senior Data Analyst"

    if "junior data analyst" in titulo:
        return "Junior Data Analyst"

    if "lead data analyst" in titulo:
        return "Lead Data Analyst"

    if "data analyst" in titulo:
        return "Data Analyst"

    return titulo.title()

In [ ]:
vacantes["familia_puesto"] = vacantes["title"].apply(normalizar_puesto)

In [ ]:
vacantes["familia_puesto"] = (
    vacantes["title"]
    .str.lower()
    .str.strip()
)

vacantes["familia_puesto"].value_counts().head(30)

In [ ]:
correcciones = {
    "data analyst": "Data Analyst",
    "data analyst ii": "Data Analyst II",
    "data analyst i": "Data Analyst I",
    "data analyst (m/w/d)": "Data Analyst"
}

vacantes["title_limpio"] = (
    vacantes["title_normalizado"]
    .replace(correcciones)
)

In [ ]:
# Convirtamos las fechas al formato corrento de las olumnas "created_at" y "published_at"
vacantes["created_at"] = pd.to_datetime(
    vacantes["created_at"],
    format="ISO8601"
)

vacantes["published_at"] = pd.to_datetime(
    vacantes["published_at"],
    format="ISO8601"
)

habilidades["created_at"] = pd.to_datetime(
    habilidades["created_at"],
    format="ISO8601"
)

In [ ]:
vacantes.info()

In [ ]:
print(vacantes["experience_level"].unique())

In [ ]:
vacantes["title_normalizado"] = (
    vacantes["title"]
    .str.lower()
    .str.strip()
)

vacantes["categoria_puesto"] = (
    vacantes["title_limpio"]
    .apply(
        lambda x: "Data Analyst" 
        if "data analyst" in x.lower()
        else x
    )
)

In [ ]:
print("Títulos originales:", vacantes["title"].nunique())
print("Títulos normalizados:", vacantes["title_normalizado"].nunique())

======================================================
Etapa 5 - Unificacion de las tablas mediante merge
======================================================

In [ ]:
vacantes_completas = vacantes.merge(
    vacantes_habilidades,
    left_on="id",
    right_on="vacancy_id",
    how="left"
)

In [ ]:
vacantes_completas = vacantes_completas.merge(
    habilidades,
    left_on="skill_id",
    right_on="id",
    how="left",
    suffixes=("", "_habilidad")
)

In [ ]:
print(vacantes_completas.shape)

vacantes_completas.head()

In [ ]:
print(vacantes_completas.columns.tolist())

======================================================
Guardamos el dataset limpio y unificado
======================================================

In [ ]:
vacantes_completas.to_csv(
    "../data/processed/vacantes_completas.csv",
    index=False
)

======================================================
Etapa 6 - EDA (Analisis exploratorio del dataset)
======================================================

Hacemos un analisis para responder preguntas concretas 

Pregunta 1: Cuales son las habilidades más demandadas?


In [ ]:

top_habilidades = (
    vacantes_completas["name"]
    .value_counts()
    .head(20)
)

top_habilidades

==========================================================================================================================================================

Verificamos en grafico las habilidades más demandadas

In [ ]:
import matplotlib.pyplot as plt

top_habilidades.plot(
    kind="barh",
    figsize=(10,7)
)

plt.title("Top 20 habilidades más demandadas")
plt.xlabel("Cantidad de ofertas")
plt.ylabel("Habilidad")

plt.tight_layout()
plt.show()

Pregunta 2: Cuales son los puestos más frecuentes?


In [ ]:
top_puestos = (
    vacantes["familia_puesto"]
    .value_counts()
    .head(15)
)

top_puestos

### ==========================================================================================================================================================

Graficamos los puestos más frecuentes para q sea mas facil observar


In [ ]:
top_puestos.plot(
    kind="barh",
    figsize=(10,6)
)

plt.title("Puestos más publicados")

plt.tight_layout()
plt.show()

==========================================================================================================================================================

Pregunta 3: Que empresas publican mas vacantes?

In [ ]:
top_empresas = (
    vacantes["company"]
    .value_counts()
    .head(15)
)

top_empresas

In [ ]:
top_empresas = (
    vacantes[vacantes["company"] != "Unknown"]["company"]
    .value_counts()
    .head(15)
)

top_empresas.plot(
    kind="bar",
    figsize=(10,6)
)

plt.title("Top 15 empresas con más vacantes publicadas")
plt.xlabel("Empresa")
plt.ylabel("Cantidad de vacantes")
plt.xticks(rotation=45, ha="right")

plt.tight_layout()
plt.show()

Podemos observar que el dataset contiene datos desconocidos para identificar objetivamente la informacion, asi que omitimos esa informacion en el grafico

### ==========================================================================================================================================================

Pregunta 4: Que nivel de experiencia buscan?

In [ ]:
vacantes["experience_level"].value_counts()

Graficamos para mas comodidad y entendimiento

In [ ]:
vacantes["experience_level"].value_counts().plot(
    kind="bar"
)

plt.title("Nivel de experiencia requerido")
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

Habilidades mas solicitadas segun experiencia

In [ ]:
def top_habilidades_por_experiencia(nivel, cantidad=10):
    return (
        vacantes_completas[
            vacantes_completas["experience_level"] == nivel
        ]["name"]
        .value_counts()
        .head(cantidad)
    )

# top_habilidades_por_experiencia("Junior")
# top_habilidades_por_experiencia("Entry level")
# top_habilidades_por_experiencia("Middle")
# top_habilidades_por_experiencia("Senior")

In [ ]:
top_habilidades_por_experiencia("Junior").plot(
    kind="barh",
    figsize=(8,5)
)

plt.title("Top habilidades - Junior")
plt.xlabel("Cantidad de ofertas")
plt.tight_layout()
plt.show()

In [ ]:
top_habilidades_por_experiencia("Entry level").plot(
    kind="barh",
    figsize=(8,5)
)

plt.title("Top habilidades - Entry level")
plt.xlabel("Cantidad de ofertas")
plt.tight_layout()
plt.show()


In [ ]:
top_habilidades_por_experiencia("Middle").plot(
    kind="barh",
    figsize=(8,5)
)

plt.title("Top habilidades - Middle")
plt.xlabel("Cantidad de ofertas")
plt.tight_layout()
plt.show()


In [ ]:
top_habilidades_por_experiencia("Senior").plot(
    kind="barh",
    figsize=(8,5)
)

plt.title("Top habilidades - Senior")
plt.xlabel("Cantidad de ofertas")
plt.tight_layout()
plt.show()


In [ ]:
top_habilidades_por_experiencia("Not specified").plot(
    kind="barh",
    figsize=(8,5)
)

plt.title("Top habilidades - Not specified")
plt.xlabel("Cantidad de ofertas")
plt.tight_layout()
plt.show()


### ==========================================================================================================================================================


Pregunta 5: Que habilidades piden mas para cada nivel?

In [ ]:

top_habilidades.plot(
    kind="barh",
    figsize=(10,6)
)

plt.title("Habilidades mas demandadas")

plt.tight_layout()
plt.show()

### ==========================================================================================================================================================

Pregunta 6: Que habilidades solicitan para el puesto mas demandado?

In [ ]:
# Vacantes del puesto Data Analyst
vacantes_data_analyst = vacantes_completas[
    vacantes_completas["title"].str.contains(
        "Data Analyst",
        case=False,
        na=False
    )
]

top_habilidades_data_analyst = (
    vacantes_data_analyst["name"]
    .value_counts()
    .head(15)
)

top_habilidades_data_analyst

In [ ]:
# graficamos las habilidades más demandadas para el puesto de Data Analyst
top_habilidades_data_analyst.plot(
    kind="barh",
    figsize=(10,6)
)

plt.title("Top 15 habilidades para puestos de Data Analyst")
plt.xlabel("Cantidad de ofertas")
plt.ylabel("Habilidad")

plt.tight_layout()
plt.show()

### Que quiere decir esto?

Se puede observa que las habilidades más solicitadas para el puesto de **Data Analyst** se concentran en tecnologías de análisis, bases de datos y visualización.

Esto permite identificar cuáles son las competencias técnicas más valoradas por las empresas al contratar analistas de datos.

**Implicancia para TalentHub:** estas tecnologías deberían formar parte del contenido principal del bootcamp, ya que incrementan la empleabilidad de los estudiantes.

### ==========================================================================================================================================================

Pregunta 7 - Qué habilidades aparecen juntas con mayor frecuencia?

In [ ]:
habilidades_por_vacante = (
    vacantes_completas
    .groupby("vacancy_id")["name"]
    .apply(list)
)

habilidades_por_vacante.head()

In [ ]:
from itertools import combinations
from collections import Counter

In [ ]:
contador = Counter()

for habilidades in habilidades_por_vacante:

    habilidades_unicas = sorted(set(habilidades))

    pares = combinations(habilidades_unicas, 2)

    contador.update(pares)


top_combinaciones = contador.most_common(15)

top_combinaciones

In [ ]:
combinaciones_df = pd.DataFrame(
    top_combinaciones,
    columns=["Combinación", "Cantidad"]
)

combinaciones_df

In [ ]:
combinaciones_df.plot(
    x="Combinación",
    y="Cantidad",
    kind="bar",
    figsize=(12,6)
)

plt.title("Combinaciones de habilidades más frecuentes")
plt.xlabel("Combinación de habilidades")
plt.ylabel("Cantidad de ofertas")
plt.xticks(rotation=45, ha="right")

plt.tight_layout()
plt.show()

### Analisis de lo q vemos aca

El análisis de coocurrencia muestra que determinadas habilidades suelen solicitarse en conjunto dentro de una misma oferta laboral.

Esto sugiere que las empresas no buscan conocimientos aislados, sino combinaciones de competencias que permitan desempeñar el rol de manera integral.

**Implicancia para TalentHub:** el bootcamp debería organizarse en módulos que agrupen tecnologías complementarias (por ejemplo, SQL junto con Python o herramientas de visualización), en lugar de enseñar cada herramienta de forma independiente.

### ==========================================================================================================================================================

###     Etapa 7: insights de los datos adquiridos del analisis 

    Hallazgo 1: SQL y Python son las habilidades más demandadas

    ¿Qué se descubrió?

El análisis de las vacantes mostró que SQL y Python se encuentran entre las habilidades técnicas más solicitadas por las empresas. Además, herramientas como Excel, Power BI y Tableau también aparecen con alta frecuencia.

    ¿Por qué es importante?

Este resultado indica que el mercado busca profesionales capaces de obtener, transformar, analizar y comunicar datos. Para TalentHub, esto implica que estas tecnologías deberían constituir el núcleo del bootcamp.

    Explicación alternativa considerada

Una posible explicación era que SQL apareciera como la habilidad más frecuente porque el conjunto de datos estuviera compuesto principalmente por ofertas relacionadas exclusivamente con bases de datos.

    ¿Cómo se evaluó?

Se revisaron los títulos de las vacantes y se observó que la mayoría corresponden a puestos de Data Analyst, donde SQL forma parte de las competencias habituales junto con Python y herramientas de visualización.

    ¿Por qué se descartó?

La presencia simultánea de SQL, Python y herramientas de análisis en un mismo tipo de puesto indica que la alta frecuencia de SQL no responde únicamente a ofertas especializadas en bases de datos, sino a las competencias requeridas para perfiles de análisis de datos.

    Hallazgo 2: Data Analyst es el perfil con mayor demanda
    
    ¿Qué se descubrió?

El puesto Data Analyst concentra la mayor cantidad de vacantes del conjunto de datos.

    ¿Por qué es importante?

Este resultado confirma que existe una fuerte demanda por este perfil y que un bootcamp orientado a formar Analistas de Datos tiene una alta alineación con el mercado laboral.

    Explicación alternativa considerada

Una posible explicación era que el predominio de Data Analyst fuera consecuencia de un sesgo en el proceso de recolección de datos.

    ¿Cómo se evaluó?

Se revisó la composición del dataset y se observó que contiene múltiples variantes del puesto (Senior, Junior, Lead, etc.) y diferentes empresas, aunque el conjunto de datos está claramente enfocado en ofertas relacionadas con análisis de datos.

    ¿Por qué no puede descartarse completamente?

No es posible descartar totalmente esta explicación porque el dataset utilizado ya estaba orientado al perfil de Data Analyst. Por lo tanto, este resultado debe interpretarse dentro del alcance del conjunto de datos analizado y no como una representación de todo el mercado tecnológico.


    Hallazgo 3: Las empresas buscan combinaciones de habilidades
    
    ¿Qué se descubrió?

El análisis de coocurrencia mostró que habilidades como SQL, Python y herramientas de visualización aparecen frecuentemente solicitadas dentro de una misma vacante.

    ¿Por qué es importante?

Esto indica que las empresas buscan profesionales capaces de integrar diferentes herramientas en un mismo flujo de trabajo y no especialistas en una única tecnología.

    Explicación alternativa considerada

Una posible explicación era que las combinaciones observadas fueran consecuencia de unas pocas ofertas con listas muy extensas de requisitos.

    ¿Cómo se evaluó?

Se agruparon las habilidades por vacante y se analizaron las combinaciones más repetidas mediante un conteo de coocurrencias.

    ¿Por qué se descartó?

Se comprobó que determinadas combinaciones aparecen de forma recurrente en numerosas ofertas distintas, lo que indica que no dependen únicamente de unas pocas publicaciones con listas extensas de habilidades.

    Hallazgo 4: La experiencia requerida se concentra en niveles iniciales e intermedios
    
    ¿Qué se descubrió?

Entre las vacantes que especifican el nivel de experiencia, predominan los perfiles Entry Level, Junior y Middle. Además, una proporción importante de las ofertas no informa este requisito.

    ¿Por qué es importante?

Este resultado sugiere que existen oportunidades para personas que buscan incorporarse al mercado laboral o dar sus primeros pasos en análisis de datos.

    Explicación alternativa considerada

Una posible explicación era que las empresas publicaran vacantes sin especificar la experiencia porque todas estuvieran dirigidas a perfiles senior.

    ¿Cómo se evaluó?

Se analizaron los valores de la columna experience_level, identificando tanto las categorías presentes como los registros sin especificar.

    ¿Por qué se descartó?

Entre las vacantes que sí informan el nivel de experiencia predominan claramente los perfiles iniciales e intermedios, por lo que no existe evidencia de que el mercado esté orientado mayoritariamente a perfiles senior.

### ==========================================================================================================================================================

### Etapa 8: Recomendaciones posteriores al analisis para TalentHub

    ° Priorizar la enseñanza de SQL, ya que aparece de forma recurrente en la mayoría de las ofertas laborales analizadas.

    ° Incorporar Python como lenguaje principal para análisis y automatización de datos.

    ° Complementar la formación con herramientas de visualización como Power BI o Tableau para fortalecer la capacidad de comunicar resultados.

    ° Diseñar el contenido del bootcamp en módulos progresivos (Fundamentos, Análisis y Visualización), reflejando la combinación de habilidades observada en las ofertas laborales.

    ° Incluir proyectos prácticos que integren SQL, Python y herramientas de visualización, ya que estas tecnologías suelen solicitarse en conjunto.

### ==============================================================================

### Etapa 10: Conclusión del analisis

    El análisis de las vacantes permitió identificar las habilidades y perfiles más demandados por el mercado laboral para posiciones relacionadas con el análisis de datos.

    Los resultados muestran que SQL, Python y herramientas de visualización constituyen el núcleo de competencias técnicas requeridas por las empresas. Asimismo, la alta demanda de perfiles Data Analyst y la frecuente combinación de estas tecnologías respaldan la necesidad de ofrecer una formación integral orientada a la empleabilidad.

    Las recomendaciones propuestas proporcionan una base para que TalentHub diseñe un bootcamp alineado con las necesidades actuales del mercado y mejorar la empleabilidad de sus futuros egresados.

### ==========================================================================================================================================================

## Limitaciones del análisis

Durante el desarrollo del proyecto se identificaron algunas limitaciones propias del conjunto de datos:

- El dataset se concentra principalmente en vacantes relacionadas con el perfil de Data Analyst, lo que limita la comparación con otros roles tecnológicos.
- No se dispone de información sobre formación académica, certificaciones, idiomas o habilidades blandas.
- Aunque existen fechas de publicación, el período analizado no permite evaluar tendencias de crecimiento a largo plazo.
- No se cuenta con información salarial, por lo que no fue posible realizar comparaciones económicas entre perfiles.